In [ ]:
import networkx as nx
import pandas as pd
import matplotlib.pyplot as plt
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.clustering import KMeans
from pyspark.ml import Pipeline
import time
import warnings

warnings.filterwarnings("ignore")

def main():
    start_time = time.time()

    # ----------------------------
    # PART 1: NETWORK ANALYSIS WITH NETWORKX
    # ----------------------------
    print("\n" + "="*50)
    print("STARTING NETWORK ANALYSIS WITH NETWORKX")
    print("="*50 + "\n")

    try:
        print("Loading nodes and edges...")
        nodes = pd.read_csv('nodes.csv')
        edges = pd.read_csv('edges.csv')
        print(f"Loaded {len(nodes)} nodes and {len(edges)} edges")
        print("Nodes columns:", nodes.columns.tolist())
        print("Edges columns:", edges.columns.tolist())

        # Drop duplicate artists
        initial_count = len(nodes)
        nodes = nodes.drop_duplicates(subset=['spotify_id'], keep='first')
        print(f"Removed {initial_count - len(nodes)} duplicate artists")

        # Handle missing values
        print("\nChecking for missing values in nodes...")
        print(nodes.isnull().sum())
        nodes['popularity'] = nodes['popularity'].fillna(nodes['popularity'].median())
        nodes['followers'] = nodes['followers'].fillna(nodes['followers'].median())

        # Use top 5000 artists by followers to reduce computation time
        nodes = nodes.sort_values(by='followers', ascending=False).head(5000)
        print(f"Using top {len(nodes)} artists by followers for graph construction.")

    except Exception as e:
        print(f"Error loading files: {e}")
        return

    print("\nCreating graph...")
    G = nx.Graph()
    node_attrs = nodes.set_index('spotify_id').to_dict('index')
    G.add_nodes_from(node_attrs.items())

    # Filter edges to only include those between top artists
    valid_ids = set(nodes['spotify_id'])
    filtered_edges = edges[edges['id_0'].isin(valid_ids) & edges['id_1'].isin(valid_ids)]
    G.add_edges_from(filtered_edges[['id_0', 'id_1']].values.tolist())

    print("\nBASIC GRAPH PROPERTIES:")
    print(f"- Nodes: {G.number_of_nodes():,}")
    print(f"- Edges: {G.number_of_edges():,}")
    print(f"- Average Degree: {sum(dict(G.degree()).values()) / G.number_of_nodes():.2f}")
    print(f"- Connected Components: {nx.number_connected_components(G)}")
    print(f"- Edge Density: {nx.density(G):.6f}")

    # Subgraph Visualization
    print("\nGenerating visualization...")
    sample_nodes = list(G.nodes())[:100]
    plt.figure(figsize=(12, 8))
    nx.draw_spring(G.subgraph(sample_nodes), with_labels=False, node_size=20, alpha=0.6)
    plt.title("Spotify Artist Collaboration Network (100 Node Subset)")
    plt.savefig('network_visualization.png')
    plt.close()

    # Degree Distribution
    degrees = [d for _, d in G.degree()]
    plt.hist(degrees, bins=50, log=True)
    plt.title("Degree Distribution (Log Scale)")
    plt.xlabel("Degree")
    plt.ylabel("Count")
    plt.savefig('degree_distribution.png')
    plt.close()

    # Centrality Calculations (Optimized)
    print("\nCalculating centrality measures (optimized)...")
    largest_cc = max(nx.connected_components(G), key=len)
    G_sub = G.subgraph(largest_cc)

    centrality_measures = {
        'degree': nx.degree_centrality(G),
        'betweenness': nx.betweenness_centrality(G, k=min(50, len(G))),
        'closeness': nx.closeness_centrality(G_sub),
        'eigenvector': nx.eigenvector_centrality(G, max_iter=200, tol=1e-3),
        'pagerank': nx.pagerank(G, alpha=0.85, max_iter=100)
    }

    try:
        diameter = nx.diameter(G_sub)
    except Exception as e:
        print(f"Could not compute diameter: {e}")
        diameter = None

    # Top Artists by Centrality
    print("\nTOP 5 ARTISTS BY CENTRALITY:")
    top_artists = {}
    for measure, values in centrality_measures.items():
        top = sorted(values.items(), key=lambda x: -x[1])[:5]
        top_artists[measure] = top
        print(f"\n{measure.upper()}:")
        for artist_id, score in top:
            print(f"- {G.nodes[artist_id].get('name', 'Unknown')}: {score:.4f}")

    # ----------------------------
    # PART 2: MACHINE LEARNING WITH PYSPARK
    # ----------------------------
    print("\n" + "="*50)
    print("STARTING MACHINE LEARNING WITH PYSPARK")
    print("="*50 + "\n")

    spark = SparkSession.builder \
        .appName("SpotifyArtistAnalysis") \
        .config("spark.driver.memory", "4g") \
        .config("spark.executor.memory", "4g") \
        .getOrCreate()

    centrality_df = pd.DataFrame({
        'artist_id': list(G.nodes()),
        'degree': [centrality_measures['degree'].get(n, 0) for n in G.nodes()],
        'betweenness': [centrality_measures['betweenness'].get(n, 0) for n in G.nodes()],
        'closeness': [centrality_measures['closeness'].get(n, 0) for n in G.nodes()],
        'eigenvector': [centrality_measures['eigenvector'].get(n, 0) for n in G.nodes()],
        'pagerank': [centrality_measures['pagerank'].get(n, 0) for n in G.nodes()]
    })

    node_attrs = pd.DataFrame.from_dict(dict(G.nodes(data=True)), orient='index')
    node_attrs.reset_index(inplace=True)
    node_attrs.rename(columns={'index': 'artist_id'}, inplace=True)

    final_df = pd.merge(node_attrs, centrality_df, on='artist_id')
    spark_df = spark.createDataFrame(final_df)

    feature_cols = ['followers', 'degree', 'betweenness', 'closeness', 'eigenvector', 'pagerank']

    # Linear Regression
    print("\nRunning linear regression...")
    assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
    scaler = StandardScaler(inputCol="features", outputCol="scaledFeatures")
    lr = LinearRegression(featuresCol="scaledFeatures", labelCol="popularity")
    pipeline = Pipeline(stages=[assembler, scaler, lr])

    train, test = spark_df.randomSplit([0.8, 0.2], seed=42)
    print(f"Training: {train.count():,}, Testing: {test.count():,}")

    model = pipeline.fit(train)
    lr_model = model.stages[-1]

    predictions = model.transform(test)
    evaluator = RegressionEvaluator(labelCol="popularity", predictionCol="prediction")
    rmse = evaluator.evaluate(predictions, {evaluator.metricName: "rmse"})
    r2 = evaluator.evaluate(predictions, {evaluator.metricName: "r2"})

    print("\nREGRESSION RESULTS:")
    print(f"- RMSE: {rmse:.4f}")
    print(f"- R^2: {r2:.4f}")
    print("\nMODEL COEFFICIENTS:")
    for col, coef in zip(feature_cols, lr_model.coefficients):
        print(f"- {col}: {coef:.6f}")
    print(f"- Intercept: {lr_model.intercept:.4f}")

    # K-Means Clustering
    print("\nRunning K-Means Clustering...")
    wssse_values = []
    k_values = [2, 3, 4, 5]

    for k in k_values:
        kmeans = KMeans(featuresCol="scaledFeatures", k=k, seed=42)
        k_pipeline = Pipeline(stages=[assembler, scaler, kmeans])
        k_model = k_pipeline.fit(spark_df)
        wssse = k_model.stages[-1].summary.trainingCost
        wssse_values.append(wssse)
        print(f"- k={k}: WSSSE={wssse:,.2f}")

    reductions = [wssse_values[i-1] - wssse_values[i] for i in range(1, len(wssse_values))]
    optimal_k = k_values[reductions.index(max(reductions)) + 1] if reductions else 3
    print(f"\nOptimal number of clusters: k={optimal_k}")

    final_kmeans = KMeans(featuresCol="scaledFeatures", k=optimal_k, seed=42)
    final_model = Pipeline(stages=[assembler, scaler, final_kmeans]).fit(spark_df)
    centers = final_model.stages[-1].clusterCenters()

    print("\nCLUSTER CENTERS:")
    for i, center in enumerate(centers):
        print(f"Cluster {i}: {center}")

    spark.stop()

    # Output Summary
    with open("results_summary.txt", "w") as f:
        f.write("SPOTIFY ARTIST NETWORK ANALYSIS RESULTS\n")
        f.write("="*50 + "\n\n")
        f.write("NETWORK PROPERTIES:\n")
        f.write(f"- Nodes: {G.number_of_nodes():,}\n")
        f.write(f"- Edges: {G.number_of_edges():,}\n")
        f.write(f"- Average Degree: {sum(dict(G.degree()).values()) / G.number_of_nodes():.2f}\n")
        f.write(f"- Connected Components: {nx.number_connected_components(G)}\n")
        f.write(f"- Edge Density: {nx.density(G):.6f}\n")
        if diameter:
            f.write(f"- Diameter: {diameter}\n")
        f.write("\nTOP 5 ARTISTS BY CENTRALITY:\n")
        for measure, artists in top_artists.items():
            f.write(f"\n{measure.upper()}:\n")
            for artist_id, score in artists:
                f.write(f"- {G.nodes[artist_id].get('name', 'Unknown')}: {score:.4f}\n")
        f.write("\nREGRESSION RESULTS:\n")
        f.write(f"- RMSE: {rmse:.4f}\n")
        f.write(f"- R^2: {r2:.4f}\n")
        f.write("\nCLUSTERING RESULTS:\n")
        f.write(f"- Optimal k: {optimal_k}\n")
        for i, center in enumerate(centers):
            f.write(f"Cluster {i} center: {center}\n")

    print("\nAll results saved to 'results_summary.txt'")
    print(f"\nCOMPLETED! Total runtime: {(time.time() - start_time) / 60:.1f} minutes")

if __name__ == "__main__":
    main()


STARTING NETWORK ANALYSIS WITH NETWORKX

Loading nodes and edges...
Loaded 156422 nodes and 300386 edges
Nodes columns: ['spotify_id', 'name', 'followers', 'popularity', 'genres', 'chart_hits']
Edges columns: ['id_0', 'id_1']
Removed 102 duplicate artists

Checking for missing values in nodes...
spotify_id         0
name               4
followers          4
popularity         0
genres             0
chart_hits    136758
dtype: int64
Using top 5000 artists by followers for graph construction.

Creating graph...

BASIC GRAPH PROPERTIES:
- Nodes: 5,000
- Edges: 27,853
- Average Degree: 11.14
- Connected Components: 687
- Edge Density: 0.002229

Generating visualization...

Calculating centrality measures (optimized)...

TOP 5 ARTISTS BY CENTRALITY:

DEGREE:
- Snoop Dogg: 0.0394
- Ty Dolla $ign: 0.0372
- Gucci Mane: 0.0362
- French Montana: 0.0342
- Lil Wayne: 0.0340

BETWEENNESS:
- Snoop Dogg: 0.0475
- Cem Adrian: 0.0326
- Diplo: 0.0325
- French Montana: 0.0319
- R3HAB: 0.0303

CLOSENESS: